### Secret Test Set Accuracy & Confusion Matrix ###
    - 64% accuracy on the secret test set

Confusion matrix:

         [[ 118 182 ]
          [ 33  267 ]]

Classification report:

              precision    recall  f1-score   support

    negative       0.78      0.39      0.52       300
    positive       0.59      0.89      0.71       300

    accuracy                           0.64       600
    macro avg      0.69      0.64      0.62       600
    weighted avg   0.69      0.64      0.62       600

### Comparison With Public Test Set Results ###

If we compare the confusion matrices for the public test set and secret test set, we can see that the proportions are very similar between the two:

        Secret Set            Public Set   
     [ 19.6%  30.3% ]      [ 20.7%  29.3% ]
     [  5.5%  44.5% ]      [  5.0%  45.0% ]

About 1.5% more of the samples were false negatives or false positives on the secret set compared to the test set, with a proportionally greater increase in false positives. This is fairly small amount and overall shows the model's performance on the public test set was a good representation of how it performs on the secret test set.

The accuracy decreased accordingly from 66% to 64% on the secret set compared to the public test set.

### What Would I Try With More Time/Compute? ###

After reflecting on the issues I had preventing the model from overfitting on the training set, I think trying to update all the GPT-2 weights on such a tiny dataset was a flawed idea. I justified it at the time because I was getting such poor performance with the transformer weights frozen, but I think freezing the weights in some form was a more profitable path to pursue. I would have liked to try keeping the weights frozen for all but the last few layers of the transformer since that is one idea that I left on the table and did not explore further. I think using a larger batch size also could have helped my performance, which I could not experiment with on my laptop since I had a memory bottleneck. Abandoning the pretrained transformer and taking advantage of preexisting word embeddings seems like a tempting choice, but I think there's a hard limit on the kinds of reviews you could properly classify with that approach.

In [1]:
import torch
import csv
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.metrics import classification_report, confusion_matrix

/home/john/Documents/Classes/Summer 2026/CYSE 499 - Machine Learning and AI/Homework 2/CYSE_Assignment2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class GPT2Classifier(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=2):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")  # NOT frozen this time
        gpt2_dim = self.gpt2.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(gpt2_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, enc):
        out = self.gpt2(**enc)
        hidden = out.last_hidden_state  # (batch, seq_len, hidden_dim)

        last_idx = enc["attention_mask"].sum(dim=1) - 1
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        pooled = hidden[batch_idx, last_idx]  # (batch, hidden_dim)

        return self.head(pooled)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def collate_fn(batch):
    ids, texts, labels = zip(*batch)
    enc = tokenizer(list(texts), return_tensors="pt", padding=True,
                     truncation=True, max_length=512)
    return list(ids), enc, torch.tensor(labels, dtype=torch.long)


class ReviewDataset(Dataset):
    def __init__(self, ids, texts, labels, max_len=512):
        self.ids = ids
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx], self.labels[idx]


def load_data(csv_path):
    ids, texts, labels = [], [], []
    with open(csv_path, encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        next(reader)  # skip header row
        for row in reader:
            ids.append(row[0])
            texts.append(row[1])
            labels.append(int(row[2]))
    return ids, texts, labels


def write_predictions_csv(ids, preds, out_path):
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "predicted_label"])
        for id_, pred in zip(ids, preds):
            writer.writerow([id_, int(pred)])
    print(f"Wrote {len(ids)} predictions to {out_path}")

def get_predictions(model, loader):
    """Run the model over a loader and return (ids, true_labels, predicted_labels)."""
    model.eval()
    all_ids, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for ids, enc, yb in tqdm(loader, desc="Evaluating model on test set", unit="batch"):
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            preds = model(enc).argmax(dim=1).cpu().numpy()
            all_ids.extend(ids)
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    return all_ids, all_labels, all_preds


BATCH_SIZE = 6
SAVE_PATH = "model_checkpoint/5_epoch_fine_tune_sentiment_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_ids, test_texts, test_labels = load_data("data/hidden_test_with_labels.csv")

test_loader = DataLoader(ReviewDataset(test_ids, test_texts, test_labels),
                              batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


model = GPT2Classifier().to(DEVICE)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))

eval_ids, eval_labels, eval_preds = get_predictions(model, test_loader)

Evaluating model on test set: 100%|██████████| 100/100 [03:36<00:00,  2.16s/batch]


### Generate Confusion Matrix & Report ###

In [3]:
print("Confusion matrix:\n")
print(confusion_matrix(eval_labels, eval_preds))
print("\nClassification report:\n")
print(classification_report(eval_labels, eval_preds, target_names=["negative", "positive"]))

Confusion matrix:

[[118 182]
 [ 33 267]]

Classification report:

              precision    recall  f1-score   support

    negative       0.78      0.39      0.52       300
    positive       0.59      0.89      0.71       300

    accuracy                           0.64       600
   macro avg       0.69      0.64      0.62       600
weighted avg       0.69      0.64      0.62       600



### Generate CSV ###

In [4]:
write_predictions_csv(eval_ids, eval_preds, "hidden_test_predictions.csv")

Wrote 600 predictions to hidden_test_predictions.csv
